
# Applied Machine Learning – DDoS Detection (CIC-DDoS2019)

This notebook implements **6 algorithms** required for the coursework.

## Machine Learning
1. Logistic Regression  
2. Random Forest  
3. Support Vector Machine (SVM)  
4. K-Nearest Neighbors (KNN)

## Deep Learning
5. Artificial Neural Network (ANN)  
6. Multilayer Perceptron (MLP)

Dataset: CIC-DDoS2019


In [ ]:

!pip install pandas numpy scikit-learn seaborn tensorflow matplotlib


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

import matplotlib.pyplot as plt
import seaborn as sns


## Upload Dataset CSV

In [ ]:

from google.colab import files
uploaded = files.upload()


In [ ]:

df = pd.read_csv("ddos_dataset.csv")
print("Dataset Shape:", df.shape)
df.head()


## Data Preprocessing

In [ ]:

df = df.dropna()
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)


In [ ]:

X = df.drop('Label', axis=1)
y = df['Label']


## Feature Scaling

In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## Train Test Split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


## Evaluation Function

In [ ]:

def evaluate_model(name, y_test, y_pred):

    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(name)
    print("Accuracy:", acc)
    print("Precision:", pre)
    print("Recall:", rec)
    print("F1 Score:", f1)

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(name + " Confusion Matrix")
    plt.show()

    return acc, pre, rec, f1


## Logistic Regression

In [ ]:

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
lr_results = evaluate_model("Logistic Regression", y_test, y_pred_lr)


## Random Forest

In [ ]:

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
rf_results = evaluate_model("Random Forest", y_test, y_pred_rf)


## Support Vector Machine

In [ ]:

svm = SVC()
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
svm_results = evaluate_model("SVM", y_test, y_pred_svm)


## K-Nearest Neighbors

In [ ]:

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
knn_results = evaluate_model("KNN", y_test, y_pred_knn)


## Artificial Neural Network (ANN)

In [ ]:

ann = Sequential()

ann.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
ann.add(Dense(32, activation='relu'))
ann.add(Dense(1, activation='sigmoid'))

ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

ann.fit(X_train, y_train, epochs=10, batch_size=64)

y_pred_ann = (ann.predict(X_test) > 0.5).astype(int)
ann_results = evaluate_model("ANN", y_test, y_pred_ann)


## Multilayer Perceptron (MLP)

In [ ]:

mlp = MLPClassifier(
    hidden_layer_sizes=(128,64,32),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=42
)

mlp.fit(X_train, y_train)

y_pred_mlp = mlp.predict(X_test)
mlp_results = evaluate_model("MLP", y_test, y_pred_mlp)


## Model Comparison

In [ ]:

results = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Random Forest",
        "SVM",
        "KNN",
        "ANN",
        "MLP"
    ],
    "Accuracy":[
        lr_results[0],
        rf_results[0],
        svm_results[0],
        knn_results[0],
        ann_results[0],
        mlp_results[0]
    ]
})

results


In [ ]:

plt.figure()
plt.bar(results["Model"], results["Accuracy"])
plt.xticks(rotation=45)
plt.title("Model Accuracy Comparison")
plt.show()
